# CDS 4003 – Logistic Regression (Python)

This notebook evaluates **Pass / Fail** predictions using the prepared `CleanedDataCDS4003.csv` dataset and **10-fold stratified cross-validation**.

**Important:** This is a Python reimplementation of the model workflow built in **Altair AI Studio**. Altair uses its own H2O operator and validation settings, so **results do not have to match**. Keep the actual Altair results in the Submission 3 report.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# 2. Load the cleaned dataset from Submission 1
# Upload CleanedDataCDS4003.csv beside this notebook first.
df = pd.read_csv("CleanedDataCDS4003.csv", sep=";")
print("Dataset shape:", df.shape)
print("Pass/Fail counts:", df["pass_status"].value_counts())
df.head()

In [ ]:
# 3. Define the features (inputs) and target (what we predict)
features = [
    "time_management_score", "previous_gpa", "attendance_percentage",
    "previous_exam_score", "assignment_completion_rate", "study_hours_per_day",
    "self_study_hours", "practice_tests_completed", "exam_preparation_days"
]
X = df[features]
y = df["pass_status"].str.strip().str.title()  # Pass or Fail
assert not X.isna().any().any(), "The cleaned features should have no missing values"
assert set(y.unique()) == {"Pass", "Fail"}

In [ ]:
# 4. Build Logistic Regression
# No regularization approximates the Altair setting "Use Regularization: off".
# Scaling is learned within each training fold to avoid additional leakage.
model = Pipeline([
    ("standardize", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        solver="lbfgs", penalty=None, max_iter=1000
    ))
])

# 10-fold CV with a reproducible Python seed.
# This is not necessarily the same fold split as Altair's "Automatic" sampling.
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=2001)

In [ ]:
# 5. Evaluate on held-out folds
results = []
predictions = np.empty(len(y), dtype=object)
for fold, (train_index, test_index) in enumerate(cv.split(X, y), start=1):
    model.fit(X.iloc[train_index], y.iloc[train_index])
    pred = model.predict(X.iloc[test_index])
    predictions[test_index] = pred
    actual = y.iloc[test_index]
    results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(actual, pred),
        "FAIL Precision": precision_score(actual, pred, pos_label="Fail"),
        "FAIL Recall": recall_score(actual, pred, pos_label="Fail"),
        "FAIL F1": f1_score(actual, pred, pos_label="Fail"),
    })
results_df = pd.DataFrame(results)
print("10-Fold Cross-Validation Results (Python):")
display(results_df.round(4))
print("\nMean scores:")
print(results_df.drop(columns="Fold").mean().map(lambda v: f"{v:.2%}"))
print("\nAccuracy variation (standard deviation):", f"{results_df['Accuracy'].std():.2%}")

In [ ]:
# 6. Confusion Matrix: FAIL is the important class
# sklearn uses actual labels for ROWS and predicted labels for COLUMNS.
labels = ["Fail", "Pass"]
cm = confusion_matrix(y, predictions, labels=labels)
print(pd.DataFrame(cm, index=["Actual Fail", "Actual Pass"],
                   columns=["Predicted Fail", "Predicted Pass"]))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    cmap="Blues", values_format="d", colorbar=False
)
plt.title("Logistic Regression – 10-Fold CV (Python)")
plt.show()

## How to interpret

- **Accuracy**: overall percentage of correct predictions.
- **FAIL Precision**: among students predicted to fail, how many really failed.
- **FAIL Recall**: among students who really failed, how many the model detected.
- **FAIL F1**: balances FAIL precision and recall.
- **Cross-validation**: trains and tests on ten different splits to check how much results vary.

**Submission 3 values from Altair (keep separate):** Accuracy **77.60% ± 0.45%**, FAIL Recall **70.57%**, FAIL Precision **50.34%**. Python values may differ.

**Important limitation:** The CSV was already normalized during earlier Altair preparation, before the folds were made. A rigorous leakage-free test should use the original unnormalized input and fit every data-dependent preprocessing step inside the training folds.